# Tamper demo — direct DB edit, bypassing the app

This notebook connects **straight to the SQLite file** the FastAPI backend uses
(`webapp/backend/data/app.db`) and edits a *data* column of one row — never a hash
column. The app never sees this write.

Then, without touching the frontend, the app's **Ledger** and **Record detail**
screens recompute every hash on their next poll (every 3–4 s), find the row no
longer matches its stored hash, and flip it **red**. If you tamper a *middle*
block, every later block goes red too (broken chain link).

**Run order for the live demo**
1. Start the backend + frontend (see `webapp/README.md`).
2. Run at least two pipelines in the UI so there's a chain with a middle block.
3. Open the **Ledger** screen and leave it visible.
4. Run the cells below to tamper an earlier row.
5. Watch the Ledger / Record detail flip to red on their own.

In [ ]:
import sqlite3, json, pathlib, textwrap

# Resolve the DB path the same way the backend does: webapp/backend/data/app.db
# (this notebook lives at webapp/tamper_demo.ipynb).
DB_PATH = (pathlib.Path.cwd() / "backend" / "data" / "app.db").resolve()
if not DB_PATH.exists():
    # fallback: search upward for it
    for parent in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        hit = parent / "webapp" / "backend" / "data" / "app.db"
        if hit.exists():
            DB_PATH = hit.resolve(); break

print("DB file:", DB_PATH)
assert DB_PATH.exists(), "app.db not found - start the backend once so it creates the DB, and run a pipeline."
conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row

## 1. Look at the current runs / chain

In [ ]:
rows = conn.execute("""
    SELECT p.id AS run_id, b.block_index, s.id AS search_id,
           s.matched_post_title, b.id AS block_id
    FROM pipeline_runs p
    JOIN blocks   b ON b.id = p.block_id
    JOIN searches s ON s.id = p.search_id
    ORDER BY b.block_index ASC
""").fetchall()

for r in rows:
    print(f"block #{r['block_index']}  run={r['run_id'][:8]}  "
          f"search={r['search_id'][:8]}  title={r['matched_post_title']!r}")
if not rows:
    print("No completed runs yet - run the pipeline in the web app first.")

## 2. Tamper an **earlier / middle** block's data

Pick a `search_id` from the list above — ideally *not* the last one, so the
cascade check lights up every later block too. We change `matched_post_title`
and leave `post_hash` untouched. That mismatch is exactly what the live
verification detects.

In [ ]:
# <-- EDIT: paste a search_id from the list above (pick an early one for the best demo)
SEARCH_ID = rows[0]["search_id"] if rows else "PASTE_SEARCH_ID_HERE"
NEW_TITLE = "FABRICATED — this caption was never the real match"

before = conn.execute(
    "SELECT matched_post_title, post_hash FROM searches WHERE id = ?", (SEARCH_ID,)
).fetchone()
print("BEFORE")
print("  title    :", before["matched_post_title"])
print("  post_hash:", before["post_hash"], "(left unchanged on purpose)")

conn.execute(
    "UPDATE searches SET matched_post_title = ? WHERE id = ?",
    (NEW_TITLE, SEARCH_ID),
)
conn.commit()

after = conn.execute(
    "SELECT matched_post_title, post_hash FROM searches WHERE id = ?", (SEARCH_ID,)
).fetchone()
print("\nAFTER")
print("  title    :", after["matched_post_title"])
print("  post_hash:", after["post_hash"], "(still the OLD hash - now inconsistent)")
print("\nNow watch the app's Ledger / Record screen flip this row (and later rows) red.")

## 3. (Optional) other columns you can tamper

Any *data* column works — never the `*_hash` / `block_hash` / `prev_hash` columns
(editing those would just hide the tamper instead of demonstrating detection).

```python
# Change the matched URL
conn.execute("UPDATE searches SET matched_post_url = ? WHERE id = ?",
             ("https://evil.example/fake", SEARCH_ID)); conn.commit()

# Corrupt the stored face encoding vector
conn.execute("UPDATE faces SET encoding_vector = ? WHERE id = ?",
             (json.dumps([0.0] * 128), "FACE_ID_HERE")); conn.commit()
```

## 4. Restore (undo the tamper)

In [ ]:
conn.execute(
    "UPDATE searches SET matched_post_title = ? WHERE id = ?",
    (before["matched_post_title"], SEARCH_ID),
)
conn.commit()
print("Restored title to:", before["matched_post_title"])
print("The app's next poll will show this row green again.")
conn.close()